# Notebook 6 — Vision Transformers: From Scratch & Pretrained

## Story
CNNs use local convolutional filters — they are inherently biased toward
spatial locality.  **Vision Transformers (ViT)** take a radically different
approach: split the image into patches and treat them as a sequence, letting
self-attention model **global** relationships across the entire image in every
layer.

We explore two flavours:

| Variant | Approach | When it works |
|---|---|---|
| **ViT from scratch** | Random init, trained entirely on our data | Needs large datasets; we expect limited performance. |
| **Pretrained ViT-B/16** | ImageNet-21k → fine-tuned | Very strong; benefits from huge pretraining data. |

Comparing both to our best CNN (from Notebook 5) answers a key question:
*does the attention mechanism give us anything the best CNN can't already provide?*

**Labels (12):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, "../")
sys.path.insert(0, "../experiments")

import torch
import torch.nn as nn
from pathlib import Path
from torchvision import models as tv_models

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, split_dataset, subsample_subset,
    get_train_transform, get_eval_transform, build_dataloaders,
    run_baselines, print_model_info,
    train_model, save_checkpoint, load_checkpoint, plot_training_history,
    plot_multi_arch_histories,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, show_saliency_examples,
    compute_multilabel_metrics, NUM_LABELS, METRIC_KEYS,
)

SEED   = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Config

In [ ]:
BASE_DATA_DIR   = "../data/aggregated"
IMAGE_SIZE      = 128      # ViT from scratch is trained at 128
BATCH_SIZE      = 128
SPLIT           = [0.8, 0.1, 0.1]
USE_SUBSET      = False
SUBSET_FRACTION = 0.1
CHECKPOINT_DIR  = Path("../checkpoints")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")

## 3. Data Loading

In [ ]:
full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)

if USE_SUBSET:
    train_raw = subsample_subset(train_raw, SUBSET_FRACTION, SEED)
    val_raw   = subsample_subset(val_raw,   SUBSET_FRACTION, SEED + 1)
    test_raw  = subsample_subset(test_raw,  SUBSET_FRACTION, SEED + 2)

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform  = get_eval_transform(IMAGE_SIZE)
train_loader, val_loader, test_loader = build_dataloaders(
    train_raw, val_raw, test_raw, train_transform, eval_transform, BATCH_SIZE
)
print(f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")

## 4. How Vision Transformers Work

1. Divide the `H×W` image into non-overlapping `P×P` patches → `N = (H/P)²` patches.
2. Linearly project each patch to a `D`-dimensional embedding.
3. Prepend a learnable `[CLS]` token and add positional encodings.
4. Pass the `N+1` token sequence through `L` Transformer encoder layers,
   each with multi-head self-attention + MLP.
5. Use the `[CLS]` token's final representation for classification.

Self-attention lets every token attend to every other token in `O(N²)` —
capturing long-range dependencies that CNNs need many layers to achieve.

## 5. Model A — ViT from Scratch

Parameters: `patch_size=16`, `embed_dim=256`, `depth=6`, `num_heads=8`,
`mlp_dim=512`.  On 128×128 images this gives 64 patches.

In [ ]:
class VisionTransformer(nn.Module):
    """Minimal ViT: 16×16 patches → 64 tokens + CLS → 6-layer Transformer → head."""

    def __init__(self, num_classes, image_size=128, patch_size=16,
                 embed_dim=256, num_heads=8, depth=6, mlp_dim=512, dropout=0.1):
        super().__init__()
        assert image_size % patch_size == 0
        self.patch_size  = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim        = 3 * patch_size * patch_size

        self.patch_embed = nn.Linear(patch_dim, embed_dim)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed   = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_dim))
        self.dropout     = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=mlp_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm        = nn.LayerNorm(embed_dim)
        self.head        = nn.Sequential(nn.Dropout(dropout), nn.Linear(embed_dim, num_classes))

    def forward(self, x):
        B, C, H, W = x.shape
        ps      = self.patch_size
        patches = x.unfold(2, ps, ps).unfold(3, ps, ps)
        patches = patches.contiguous().view(B, C, -1, ps, ps)
        patches = patches.permute(0, 2, 1, 3, 4).flatten(2)   # (B, N, patch_dim)
        x       = self.patch_embed(patches)
        cls     = self.cls_token.expand(B, -1, -1)
        x       = torch.cat((cls, x), dim=1)
        x       = self.dropout(x + self.pos_embed)
        x       = self.transformer(x)
        return self.head(self.norm(x[:, 0]))


def create_vit_scratch(num_labels: int) -> nn.Module:
    return VisionTransformer(num_classes=num_labels, image_size=IMAGE_SIZE)


print_model_info(create_vit_scratch(NUM_LABELS).to(DEVICE))

## 6. Train ViT from Scratch

ViTs trained from scratch require lower LRs and cosine annealing to converge.
We use a longer warmup (10 epochs) to stabilise training.

In [ ]:
VIT_PATH = CHECKPOINT_DIR / "vit_scratch.pth"

best_state_vit, best_f1_vit, history_vit, epochs_vit = train_model(
    create_vit_scratch, NUM_LABELS, train_loader, val_loader, DEVICE,
    lr=3e-4, weight_decay=1e-4, max_epochs=60, warmup_epochs=10,
)
save_checkpoint(best_state_vit, VIT_PATH)
print(f"\nViT (scratch) best val F1: {best_f1_vit:.4f}")
plot_training_history(history_vit, epochs_vit, "ViT (scratch)", 3e-4, 1e-4)

## 7. Model B — Pretrained ViT-B/16 (Transfer Learning)

Torchvision's `vit_b_16` was pretrained on ImageNet-1k.  We replace
the head and fine-tune the whole model with a small LR.

In [ ]:
# Pretrained ViT uses 224×224, so we need a new dataloader at that size
from utils import get_train_transform as _train_t, get_eval_transform as _eval_t
from torch.utils.data import DataLoader
from utils import TransformSubset

VIT_IMAGE_SIZE = 224
vit_train_ds = TransformSubset(train_raw, transform=_train_t(VIT_IMAGE_SIZE))
vit_val_ds   = TransformSubset(val_raw,   transform=_eval_t(VIT_IMAGE_SIZE))
vit_test_ds  = TransformSubset(test_raw,  transform=_eval_t(VIT_IMAGE_SIZE))

vit_train_loader = DataLoader(vit_train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
vit_val_loader   = DataLoader(vit_val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
vit_test_loader  = DataLoader(vit_test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


def create_vit_pretrained(num_labels: int) -> nn.Module:
    m = tv_models.vit_b_16(weights=tv_models.ViT_B_16_Weights.IMAGENET1K_V1)
    # Freeze everything except the encoder's last block and the head
    for p in m.parameters():
        p.requires_grad = False
    for p in m.encoder.layers[-2:].parameters():
        p.requires_grad = True
    in_feat = m.heads.head.in_features
    m.heads.head = nn.Linear(in_feat, num_labels)
    for p in m.heads.parameters():
        p.requires_grad = True
    return m


print_model_info(create_vit_pretrained(NUM_LABELS).to(DEVICE))

In [ ]:
VIT_TL_PATH = CHECKPOINT_DIR / "vit_pretrained.pth"

best_state_vit_tl, best_f1_vit_tl, history_vit_tl, epochs_vit_tl = train_model(
    create_vit_pretrained, NUM_LABELS, vit_train_loader, vit_val_loader, DEVICE,
    lr=1e-4, weight_decay=1e-4, max_epochs=20, warmup_epochs=3,
)
save_checkpoint(best_state_vit_tl, VIT_TL_PATH)
print(f"\nViT-B/16 (pretrained) best val F1: {best_f1_vit_tl:.4f}")
plot_training_history(history_vit_tl, epochs_vit_tl, "ViT-B/16 (pretrained)", 1e-4, 1e-4)

## 8. Compare ViT Variants

In [ ]:
# Compare on 128-resolution test set (ViT scratch) for fair comparison
entries = [
    ("ViT scratch (128)", create_vit_scratch, VIT_PATH, test_loader),
]
# ViT pretrained uses 224 resolution
entries.append(("ViT-B/16 pretrained", create_vit_pretrained, VIT_TL_PATH, vit_test_loader))

print(f"{'Model':<24} {'loss':>7} {'exact':>7} {'hamming':>8} {'IoU':>7} {'prec':>7} {'rec':>7} {'F1':>7}")
print("-" * 88)
for name, fn, ckpt, loader in entries:
    m = load_checkpoint(fn, NUM_LABELS, ckpt, DEVICE)
    imgs, lbls, preds, probs = collect_test_predictions(m, loader, DEVICE)
    all_logits = []
    m.eval()
    with torch.no_grad():
        for images, _ in loader:
            all_logits.append(m(images.to(DEVICE)).cpu())
    met = compute_multilabel_metrics(lbls, preds, logits=torch.cat(all_logits))
    print(f"{name:<24} {met['loss']:>7.4f} {met['exact_match']:>7.4f} "
          f"{met['hamming_acc']:>8.4f} {met['mean_iou']:>7.4f} "
          f"{met['precision_micro']:>7.4f} {met['recall_micro']:>7.4f} "
          f"{met['f1_micro']:>7.4f}")

## 9. Analysis — Pretrained ViT

In [ ]:
model = load_checkpoint(create_vit_pretrained, NUM_LABELS, VIT_TL_PATH, DEVICE)
images, labels, preds, probs = collect_test_predictions(model, vit_test_loader, DEVICE)
correct_idx, partial_idx, incorrect_idx = categorize_predictions(labels, preds)

show_prediction_examples(correct_idx,   images, labels, preds, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   images, labels, preds, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, images, labels, preds, "Fully Incorrect",   n=4)

In [ ]:
plot_per_class_metrics(labels, preds)
plot_confusion_matrices(labels, preds)

## 10. Conclusions

- **ViT from scratch** struggles — Transformers rely heavily on scale (data
  and compute) to outcompete CNNs; on small datasets they are at a disadvantage.
- **Pretrained ViT-B/16** is highly competitive with the best CNN backbones,
  demonstrating that attention-based global context is genuinely useful for
  multi-label recognition.
- The pretrained ViT benefits from the massive ImageNet-21k pretraining corpus
  that was not available to our scratch models.

**Next:** We identify the most impressive model overall and explore targeted
techniques to push its performance further (Notebook 7).